# BCI COG Workshop — R Exercises

**Hour 2 — Hands-on (R group)**

Work through the 5 exercises below. Each builds on the previous.
Run cells top-to-bottom. Lines marked `# TODO` are for you to complete.

**Required R packages:** `httr`, `jsonlite`, `stringr`, `terra`  
Install if missing:
```r
install.packages(c("httr", "jsonlite", "stringr", "terra"), repos = "https://cloud.r-project.org")
```

---
**Ask for help anytime!**

In [ ]:
# ── Configuration — do not modify ────────────────────────────────────────────
STAC_API_URL  <- "https://kanopia.org/stac-fastapi-pgstac/api/v1/pgstac"
COLLECTION_ID <- "2024_bci"

# Workshop credentials
COG_USER <- "panama"
COG_PASS <- "panama123"

# Default point of interest: approximate centre of BCI island
# Change these to any lon/lat inside BCI to explore a different spot
DEFAULT_LON <- -79.8450
DEFAULT_LAT <-   9.1540

cat("Config ready.\n")

In [ ]:
# ── Install and load packages ─────────────────────────────────────────────────
needed  <- c("httr", "jsonlite", "stringr", "terra")
missing <- needed[!sapply(needed, requireNamespace, quietly = TRUE)]
if (length(missing) > 0) {
  cat("Installing missing packages:", paste(missing, collapse = ", "), "\n")
  install.packages(missing, repos = "https://cloud.r-project.org")
}

library(httr)
library(jsonlite)
library(stringr)
library(terra)

# Shared helpers
`%||%` <- function(x, y) if (is.null(x) || length(x) == 0) y else x

add_auth <- function(url, user, pwd) {
  # Embed Basic Auth credentials into a URL
  if (grepl("@", url)) return(url)
  sub("://", paste0("://", user, ":", pwd, "@"), url, fixed = TRUE)
}

fmt_date <- function(date_int) {
  d <- as.character(date_int)
  paste0(substr(d,1,4), "-", substr(d,5,6), "-", substr(d,7,8))
}

cat("Packages and helpers ready.\n")

---
## Exercise 1 — Query the STAC API

**Goal:** POST a search request to the Kanopia STAC API and collect all BCI whole-island RGB COG assets.

**Hint:** use `httr::POST` to `paste0(STAC_API_URL, "/search")` with a JSON body containing `collections` and `limit`.

In [ ]:
# Exercise 1 ──────────────────────────────────────────────────────────────────
search_url <- paste0(STAC_API_URL, "/search")

# TODO: POST to the STAC search endpoint
# body should be: list(collections = list(COLLECTION_ID), limit = 500)
resp <- httr::POST(
  url    = search_url,
  body   = ...,      # list(collections = list(COLLECTION_ID), limit = 500)
  encode = "json"
)
httr::stop_for_status(resp)

data     <- httr::content(resp, as = "text", encoding = "UTF-8")
features <- jsonlite::fromJSON(data, simplifyVector = FALSE)$features
cat("Items returned by STAC:", length(features), "\n")

# Filter for whole-island RGB COG hrefs accessible via /share/1
pat      <- "(\\d{8})_bciwhole_.*rgb\\.cog\\.tif"
cog_list <- list()
idx      <- 1L

for (f in features) {
  assets  <- f$assets %||% list()
  item_id <- f$id %||% NA_character_
  for (key in names(assets)) {
    href <- assets[[key]]$href %||% ""
    m    <- regmatches(href, regexpr(pat, href, perl = TRUE))
    if (length(m) > 0 && grepl("/share/1", href, fixed = TRUE)) {  # keep only accessible assets
      date_str <- substr(m, 1, 8)
      cog_list[[idx]] <- list(date = as.integer(date_str), href = href, item_id = item_id)
      idx <- idx + 1L
    }
  }
}

# Sort by date
cog_list <- cog_list[order(sapply(cog_list, `[[`, "date"))]

cat("Found", length(cog_list), "BCI whole-island RGB COGs:\n")
for (c in cog_list) cat(" ", fmt_date(c$date), " ", c$item_id, "\n")

---
## Exercise 2 — Open a COG and display an RGB chip

**Goal:** open one COG URL with `terra::rast()`, crop a window around a point, and plot it.

**Hints:**
- Use `add_auth(href, COG_USER, COG_PASS)` to embed credentials.
- Use `/vsicurl/` prefix: `paste0("/vsicurl/", url)`.
- `terra::rast()` reads COG metadata lazily (no full download).
- Reproject your point to the COG CRS with `terra::project()`.

In [ ]:
# Exercise 2 ──────────────────────────────────────────────────────────────────
target <- cog_list[[1]]   # first date

# TODO: build the authenticated /vsicurl/ URL
vsi_url <- paste0("/vsicurl/", add_auth(target$href, COG_USER, COG_PASS))

cat("Opening COG:", fmt_date(target$date), "\n")

# TODO: open the raster with terra::rast()
r <- terra::rast(...)     # vsi_url

cat("CRS     :", terra::crs(r, describe = TRUE)$code, "\n")
cat("Bands   :", terra::nlyr(r), "\n")
cat("Size    :", terra::ncol(r), "x", terra::nrow(r), "\n")

# Reproject the point of interest to COG CRS
pt_wgs84 <- terra::vect(matrix(c(DEFAULT_LON, DEFAULT_LAT), ncol = 2),
                        crs = "EPSG:4326")
pt_cog   <- terra::project(pt_wgs84, terra::crs(r))
cx       <- terra::geom(pt_cog)[, "x"]
cy       <- terra::geom(pt_cog)[, "y"]

# Define a 512x512 px crop window (in map units)
res_x  <- terra::res(r)[1]
res_y  <- terra::res(r)[2]
half_x <- res_x * 256
half_y <- res_y * 256
win    <- terra::ext(cx - half_x, cx + half_x, cy - half_y, cy + half_y)

# TODO: crop the raster to the window
r_chip <- terra::crop(r, win)

# TODO: plot as RGB using terra::plotRGB
terra::plotRGB(...)       # r_chip, r=1, g=2, b=3, stretch="lin"
title(paste0("RGB chip — ", fmt_date(target$date)))

---
## Exercise 3 — Compute and plot the Green Leaf Index (GLI)

**Formula:** `GLI = (2G − R − B) / (2G + R + B)`

- High GLI → healthy vegetation (green in RdYlGn)
- Low GLI → dead/bare/stressed (red in RdYlGn)

In [ ]:
# Exercise 3 ──────────────────────────────────────────────────────────────────
eps <- 1e-6

# Extract individual bands
R_band <- terra::as.array(r_chip[[1]])[,,1]   # matrix
G_band <- terra::as.array(r_chip[[2]])[,,1]
B_band <- terra::as.array(r_chip[[3]])[,,1]

# TODO: compute GLI
GLI <- ...    # (2*G_band - R_band - B_band) / (2*G_band + R_band + B_band + eps)

cat("GLI range: [", min(GLI, na.rm=TRUE), ",", max(GLI, na.rm=TRUE), "]\n")

# Make a colour palette matching RdYlGn
pal <- colorRampPalette(c("#d73027","#f46d43","#fdae61",
                           "#fee08b","#d9ef8b","#a6d96a",
                           "#66bd63","#1a9850"))(256)

# Plot: RGB | GLI
par(mfrow = c(1, 2), mar = c(1,1,2,1))

terra::plotRGB(r_chip, r=1, g=2, b=3, stretch="lin",
              main = paste0("RGB — ", fmt_date(target$date)))

# TODO: plot GLI as image with the pal palette
image(...)    # t(GLI[nrow(GLI):1,]), col=pal, zlim=c(-0.5,0.5),
              # main="GLI — Green Leaf Index", xaxt="n", yaxt="n"
              # (use t() + row reversal to match raster orientation)

par(mfrow = c(1, 1))

---
## Exercise 4 — Compute VARI and GCC; compare all three indices

| Index | Formula |
|-------|---------|
| **VARI** | `(G − R) / (G + R − B)` — clip to [-1.5, 1.5] |
| **GCC**  | `G / (R + G + B)` |

**Goal:** 1×4 plot: RGB, GLI, VARI, GCC.

In [ ]:
# Exercise 4 ──────────────────────────────────────────────────────────────────
# TODO: compute VARI and GCC
VARI <- ...   # (G_band - R_band) / (G_band + R_band - B_band + eps)
VARI <- pmax(pmin(VARI, 1.5), -1.5)   # clip outliers

GCC  <- ...   # G_band / (R_band + G_band + B_band + eps)

cat("VARI range: [", min(VARI, na.rm=TRUE), ",", max(VARI, na.rm=TRUE), "]\n")
cat("GCC  range: [", min(GCC,  na.rm=TRUE), ",", max(GCC,  na.rm=TRUE), "]\n")

par(mfrow = c(1, 4), mar = c(1,1,2,1))

terra::plotRGB(r_chip, r=1, g=2, b=3, stretch="lin",
              main = paste0("RGB\n", fmt_date(target$date)))

# Helper to flip matrix for image()
flip_m <- function(m) t(m[nrow(m):1,])

image(flip_m(GLI),  col=pal, zlim=c(-0.5, 0.5),  main="GLI",  xaxt="n", yaxt="n")

# TODO: plot VARI
image(...)    # flip_m(VARI), col=pal, zlim=c(-0.5,0.8), main="VARI", xaxt="n", yaxt="n"

# TODO: plot GCC
image(...)    # flip_m(GCC),  col=pal, zlim=c(0.28,0.45), main="GCC",  xaxt="n", yaxt="n"

par(mfrow = c(1, 1))

---
## Exercise 5 (stretch) — Compare GLI across two dates

**Goal:** read the same chip from the **first** and **last** available dates.
Plot a 2×2 grid: RGB + GLI for each date.

What changed? Can you spot trees that got greener or redder?

In [ ]:
# Exercise 5 ──────────────────────────────────────────────────────────────────
date_early <- cog_list[[1]]
date_late  <- cog_list[[length(cog_list)]]

read_chip <- function(cog_entry) {
  url <- paste0("/vsicurl/", add_auth(cog_entry$href, COG_USER, COG_PASS))
  r   <- terra::rast(url)
  # Reproject point
  pt  <- terra::project(pt_wgs84, terra::crs(r))
  cx  <- terra::geom(pt)[, "x"]
  cy  <- terra::geom(pt)[, "y"]
  rx  <- terra::res(r)[1]
  ry  <- terra::res(r)[2]
  ext <- terra::ext(cx - rx*256, cx + rx*256, cy - ry*256, cy + ry*256)
  terra::crop(r, ext)
}

compute_gli_from_chip <- function(chip) {
  R <- terra::as.array(chip[[1]])[,,1]
  G <- terra::as.array(chip[[2]])[,,1]
  B <- terra::as.array(chip[[3]])[,,1]
  (2*G - R - B) / (2*G + R + B + 1e-6)
}

cat("Reading early chip:", fmt_date(date_early$date), "...\n")
# TODO: call read_chip for date_early
chip_e <- ...     # read_chip(date_early)

cat("Reading late  chip:", fmt_date(date_late$date), "...\n")
# TODO: call read_chip for date_late
chip_l <- ...     # read_chip(date_late)

gli_e <- compute_gli_from_chip(chip_e)
gli_l <- compute_gli_from_chip(chip_l)

# TODO: make a 2x2 figure
#   [row1, col1] RGB early   [row1, col2] RGB late
#   [row2, col1] GLI early   [row2, col2] GLI late
par(mfrow = c(2, 2), mar = c(1,1,3,1))

terra::plotRGB(chip_e, r=1, g=2, b=3, stretch="lin",
              main = paste0("RGB — ", fmt_date(date_early$date)))
terra::plotRGB(chip_l, r=1, g=2, b=3, stretch="lin",
              main = paste0("RGB — ", fmt_date(date_late$date)))

# TODO: plot gli_e and gli_l
image(flip_m(gli_e), col=pal, zlim=c(-0.5, 0.5),
      main=paste0("GLI — ", fmt_date(date_early$date)), xaxt="n", yaxt="n")
image(flip_m(gli_l), col=pal, zlim=c(-0.5, 0.5),
      main=paste0("GLI — ", fmt_date(date_late$date)),  xaxt="n", yaxt="n")

par(mfrow = c(1, 1))

---
## Well done!

You've just:
- Queried a STAC API for drone imagery from R
- Read sub-second chips from multi-GB COG files using `terra`
- Computed GLI, VARI, and GCC vegetation indices
- Detected temporal change in canopy health

**Next steps to explore:**
- Change `DEFAULT_LON` / `DEFAULT_LAT` to explore different parts of BCI
- Loop over all dates in `cog_list`, compute mean GLI per chip, and plot the time series
- Use `terra::writeRaster()` to save a GLI layer as a GeoTIFF